In [6]:
# Cell 1: Title + Data
print("GPU-Accelerated Heart Disease Prediction")
import pandas as pd
df = pd.read_csv("/kaggle/input/heart-disease-data/heart_disease_uci.csv")
print(f" Dataset: {df.shape} | Missing: {df.isnull().sum().sum()}")
print(df.head())


GPU-Accelerated Heart Disease Prediction
 Dataset: (920, 16) | Missing: 1759
   id  age     sex    dataset               cp  trestbps   chol    fbs  \
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True   
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False   
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False   
3   4   37    Male  Cleveland      non-anginal     130.0  250.0  False   
4   5   41  Female  Cleveland  atypical angina     130.0  204.0  False   

          restecg  thalch  exang  oldpeak        slope   ca  \
0  lv hypertrophy   150.0  False      2.3  downsloping  0.0   
1  lv hypertrophy   108.0   True      1.5         flat  3.0   
2  lv hypertrophy   129.0   True      2.6         flat  2.0   
3          normal   187.0  False      3.5  downsloping  0.0   
4  lv hypertrophy   172.0  False      1.4    upsloping  0.0   

                thal  num  
0       fixed defect    0  
1             normal    2  
2  reversable d

In [7]:
# Cell 2: Quick EDA
print("Quick EDA")
print(df.describe())
print("\nTarget distribution:")
print(df.iloc[:,-1].value_counts())


Quick EDA
               id         age    trestbps        chol      thalch     oldpeak  \
count  920.000000  920.000000  861.000000  890.000000  865.000000  858.000000   
mean   460.500000   53.510870  132.132404  199.130337  137.545665    0.878788   
std    265.725422    9.424685   19.066070  110.780810   25.926276    1.091226   
min      1.000000   28.000000    0.000000    0.000000   60.000000   -2.600000   
25%    230.750000   47.000000  120.000000  175.000000  120.000000    0.000000   
50%    460.500000   54.000000  130.000000  223.000000  140.000000    0.500000   
75%    690.250000   60.000000  140.000000  268.000000  157.000000    1.500000   
max    920.000000   77.000000  200.000000  603.000000  202.000000    6.200000   

               ca         num  
count  309.000000  920.000000  
mean     0.676375    0.995652  
std      0.935653    1.142693  
min      0.000000    0.000000  
25%      0.000000    0.000000  
50%      0.000000    1.000000  
75%      1.000000    2.000000  
max 

In [10]:
# Cell 3: Production Pipeline (Your exact features)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# YOUR exact features from notebook
numeric_features = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

X = df.drop(columns=['num']).select_dtypes(include=['number', 'object'])
y = (df['num'] > 0).astype(int)

print(f"Features: {X.shape} | Target: {y.value_counts().to_dict()}")

# Numeric pipeline (median + scale)
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

# Categorical pipeline (mode + onehot)
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

# ColumnTransformer (YOUR original!)
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Full pipeline
clf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))])

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"✅ Logistic Regression: {accuracy_score(y_test, y_pred):.1%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))


Features: (920, 15) | Target: {1: 509, 0: 411}
✅ Logistic Regression: 84.2%

Classification Report:
              precision    recall  f1-score   support

  No Disease       0.84      0.79      0.82        82
     Disease       0.84      0.88      0.86       102

    accuracy                           0.84       184
   macro avg       0.84      0.84      0.84       184
weighted avg       0.84      0.84      0.84       184



In [11]:
# Cell 4: Results Summary
print("🎉 SIMPLE PIPELINE SUCCESS!")
print("• Dataset: UCI Heart Disease")
print("• Accuracy: {:.1%}".format(accuracy_score(y_test, y_pred)))
print("• Ready for: CV → RF → CUDA speedup!")


🎉 SIMPLE PIPELINE SUCCESS!
• Dataset: UCI Heart Disease
• Accuracy: 84.2%
• Ready for: CV → RF → CUDA speedup!
